# Radiator feasibility: how big and heavy?

Is thermal rejection a feasibility-killer for an orbital data center? This notebook sizes radiators for ~1 MW of waste heat with the `orbitdc.thermal` co-design module, and shows where optimistic assumptions (BOL coatings, two-sided credit, deep-space view) quietly fail.

Net rejection is emitted radiation **minus** absorbed solar, albedo, and Earth IR — not a free cold sink.

In [ ]:
from orbitdc.thermal import thermal_codesign
from orbitdc.thermal import validation as v
from orbitdc.thermal.catalog import get_chip_stack, get_coating, get_coolant, get_radiator_surface
from orbitdc.thermal.presets import get_environment
from orbitdc.thermal.radiation import required_area_m2
from orbitdc.thermal.surfaces import RadiatorSurface

Q_MW = 1.0e6  # 1 MW of waste heat

## Idealized sanity table (no environmental absorption)
The optimistic bound: one-sided, eps=0.9, perfect deep-space view.

In [ ]:
for T in (300, 320, 350, 400, 500, 600):
    print(f"  T_rad={T} K  ->  {v.ideal_area_for_mw(T):7.0f} m^2 for 1 MW")

## Realistic area vs coating and orientation (EOL)
Subtract the absorbed environment and degrade coatings to end of life.

In [ ]:
env = get_environment("conservative_leo")
for coating_key in ("osr", "white_z93"):
    for sides in (1, 2):
        surf = RadiatorSurface(coating=get_coating(coating_key), sides=sides, max_temp_k=350.0)
        area = required_area_m2(Q_MW, 330.0, surf, env, eol=True)
        print(f"  {coating_key:9s} sides={sides}  T_rad=330K  ->  {area:8.0f} m^2 for 1 MW (EOL)")

## Coupled co-design for the demo cluster
Chip power sets the radiator temperature ceiling; size area and mass at EOL.

In [ ]:
stack = get_chip_stack("h100_direct_liquid", chip_power_w=700.0)
coolant = get_coolant("ammonia_single_phase")
surface = get_radiator_surface("deployable_osr")
res = thermal_codesign(
    q_waste_w=1.11e6,
    chip_stack=stack,
    coolant=coolant,
    surface=surface,
    env=env,
    area_available_m2=2880.0,
)
print(f"T_rad        = {res.t_rad_k:.0f} K (junction-capped: {res.junction_capped})")
print(f"T_junction   = {res.t_junction_k:.0f} K")
print(f"net flux     = {res.net_flux_w_m2:.0f} W/m^2  (env absorbs {res.absorbed_fraction:.0%})")
print(f"area required= {res.area_required_m2:.0f} m^2   ({res.m2_per_kw:.2f} m^2/kW)")
print(f"thermal mass = {res.thermal_mass_kg / 1000:.1f} t   ({res.kg_per_kw:.1f} kg/kW)")
print(f"pump power   = {res.pump_power_w / 1000:.0f} kW   bottleneck: {res.bottleneck}")
for w in res.warnings:
    print(f"  caveat: {w}")

## The optimism gap: BOL vs EOL
Beginning-of-life closure understates the radiator you actually need.

In [ ]:
surf = get_radiator_surface("deployable_osr")
bol = required_area_m2(Q_MW, 330.0, surf, env, eol=False)
eol = required_area_m2(Q_MW, 330.0, surf, env, eol=True)
print(
    f"BOL area: {bol:.0f} m^2   EOL area: {eol:.0f} m^2   (+{(eol / bol - 1) * 100:.0f}% over life)"
)

## Validation anchors
Flown hardware and a published optimistic claim.

In [ ]:
for a in v.ANCHORS:
    print(f"  {a.name}: {a.note}")